# Diebold-Mariano Test for Equal Predictive Ability

The **Diebold-Mariano (DM) test** (Diebold & Mariano, 1995) is the workhorse of forecast comparison. It tests whether two competing forecasts have equal predictive accuracy, measured by a given loss function.

This notebook covers:
1. The DM framework and hypotheses
2. Bilateral (two-sided) tests
3. Pairwise comparison matrix
4. One-sided tests for forecast superiority
5. Harvey-Leybourne-Newbold (1997) small-sample correction
6. Multi-horizon DM tests

**References:**
- Diebold, F.X. & Mariano, R.S. (1995). "Comparing Predictive Accuracy." *Journal of Business & Economic Statistics*, 13(3), 253-263.
- Harvey, D., Leybourne, S. & Newbold, P. (1997). "Testing the equality of prediction mean squared errors." *International Journal of Forecasting*, 13(2), 281-291.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

# Add parent dirs so helpers are importable
sys.path.insert(0, os.path.join(os.path.dirname("__file__"), "..", ".."))

from forecastbox.evaluation import diebold_mariano, DMResult
from utils.helpers import load_inflation_forecasts, load_m4_sample

## 1. The Diebold-Mariano Framework

Given two forecast sequences $\{\hat{y}_{1,t}\}$ and $\{\hat{y}_{2,t}\}$ for actual values $\{y_t\}$, define the **loss differential**:

$$d_t = L(e_{1,t}) - L(e_{2,t})$$

where $e_{i,t} = y_t - \hat{y}_{i,t}$ and $L(\cdot)$ is a loss function (e.g., squared error, absolute error).

**Hypotheses:**
- $H_0: E[d_t] = 0$ — the two forecasts have equal predictive accuracy
- $H_1: E[d_t] \neq 0$ — the forecasts differ in accuracy (two-sided)

**DM statistic:**

$$DM = \frac{\bar{d}}{\sqrt{\hat{V}(\bar{d})}} \xrightarrow{d} N(0, 1)$$

where $\hat{V}(\bar{d})$ is estimated using HAC (Newey-West) variance to account for serial correlation in multi-step forecasts.

In [ ]:
# Load inflation forecast data
df = load_inflation_forecasts()
print(f"Dataset: {df.shape[0]} observations, {df.shape[1]} columns")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"\nColumns: {list(df.columns)}")
df.head()

# Extract arrays
actual = df["actual"].values
fc_arima = df["fc_arima"].values
fc_ets = df["fc_ets"].values
fc_var = df["fc_var"].values
fc_naive = df["fc_naive"].values
fc_drift = df["fc_drift"].values

model_names = ["ARIMA", "ETS", "VAR", "Naive", "Drift"]
forecasts = [fc_arima, fc_ets, fc_var, fc_naive, fc_drift]

# Compute and visualize loss differentials (ARIMA vs ETS)
d_arima_ets = (actual - fc_arima)**2 - (actual - fc_ets)**2

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df.index, d_arima_ets, linewidth=0.8)
axes[0].axhline(0, color="red", linestyle="--", alpha=0.7)
axes[0].set_title("Loss Differential: ARIMA vs ETS (squared loss)")
axes[0].set_ylabel("$d_t = e_{ARIMA}^2 - e_{ETS}^2$")

axes[1].hist(d_arima_ets, bins=25, edgecolor="black", alpha=0.7)
axes[1].axvline(np.mean(d_arima_ets), color="red", linestyle="--", label=f"mean = {np.mean(d_arima_ets):.4f}")
axes[1].set_title("Distribution of Loss Differentials")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"\nMean loss differential (d_bar): {np.mean(d_arima_ets):.6f}")
print(f"Negative d_bar => ARIMA has lower average squared loss" if np.mean(d_arima_ets) < 0 else "Positive d_bar => ETS has lower average squared loss")

## 2. DM Test: ARIMA vs ETS

We start with a **two-sided test** comparing ARIMA and ETS forecasts. The null hypothesis is that both models have equal predictive accuracy under squared error loss.

In [ ]:
# Two-sided DM test: ARIMA vs ETS
result = diebold_mariano(actual, fc_arima, fc_ets, h=1, loss="mse")

print("Diebold-Mariano Test: ARIMA vs ETS")
print("=" * 50)
print(f"DM statistic:     {result.statistic:.4f}")
print(f"p-value:          {result.pvalue:.4f}")
print(f"Mean loss diff:   {result.mean_loss_diff:.6f}")
print(f"HLN corrected:    {result.hln_corrected}")
print(f"One-sided:        {result.one_sided}")
print(f"\n{result.conclusion()}")

# Also test with MAE loss
result_mae = diebold_mariano(actual, fc_arima, fc_ets, h=1, loss="mae")
print(f"\nWith MAE loss: DM={result_mae.statistic:.4f}, p={result_mae.pvalue:.4f}")
print(result_mae.conclusion())

## 3. DM Test: All Pairwise Comparisons

When comparing $K$ models, we compute an $K \times K$ matrix of DM p-values. Entry $(i, j)$ gives the p-value for testing $H_0$: model $i$ and model $j$ have equal accuracy. This provides a complete picture of relative forecast performance.

In [ ]:
# Build NxN pairwise DM p-value matrix
n_models = len(model_names)
pvalue_matrix = np.ones((n_models, n_models))
stat_matrix = np.zeros((n_models, n_models))

for i in range(n_models):
    for j in range(n_models):
        if i != j:
            res = diebold_mariano(actual, forecasts[i], forecasts[j], h=1, loss="mse")
            pvalue_matrix[i, j] = res.pvalue
            stat_matrix[i, j] = res.statistic

# Display as DataFrame
pval_df = pd.DataFrame(pvalue_matrix, index=model_names, columns=model_names)
print("Pairwise DM Test p-values (two-sided, MSE loss)")
print("=" * 60)
print(pval_df.round(4).to_string())

# Heatmap visualization
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(pvalue_matrix, cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xticks(range(n_models))
ax.set_yticks(range(n_models))
ax.set_xticklabels(model_names)
ax.set_yticklabels(model_names)

for i in range(n_models):
    for j in range(n_models):
        color = "white" if pvalue_matrix[i, j] < 0.3 else "black"
        ax.text(j, i, f"{pvalue_matrix[i, j]:.3f}", ha="center", va="center", color=color, fontsize=10)

plt.colorbar(im, label="p-value")
ax.set_title("DM Pairwise p-value Matrix (MSE loss)")
plt.tight_layout()
plt.show()

# Identify significant differences at 5%
print("\nSignificant differences (p < 0.05):")
for i in range(n_models):
    for j in range(i + 1, n_models):
        if pvalue_matrix[i, j] < 0.05:
            better = model_names[i] if stat_matrix[i, j] < 0 else model_names[j]
            print(f"  {model_names[i]} vs {model_names[j]}: p={pvalue_matrix[i, j]:.4f} ({better} is better)")

## 4. One-Sided Tests

A **one-sided DM test** is appropriate when we have a directional hypothesis — e.g., "ARIMA is *better than* the naive benchmark."

With `one_sided=True`, the alternative hypothesis becomes $H_1: E[d_t] < 0$ (forecast 1 is more accurate than forecast 2).

In [ ]:
# One-sided test: Is ARIMA better than Naive?
result_one = diebold_mariano(actual, fc_arima, fc_naive, h=1, loss="mse", one_sided=True)

print("One-Sided DM Test: ARIMA better than Naive?")
print("=" * 50)
print(f"DM statistic:   {result_one.statistic:.4f}")
print(f"p-value:        {result_one.pvalue:.4f}")
print(f"Mean loss diff: {result_one.mean_loss_diff:.6f}")
print(f"\n{result_one.conclusion()}")

# Compare all models against Naive (one-sided)
print("\n\nOne-Sided Tests: Each Model vs Naive")
print("-" * 50)
for name, fc in zip(model_names[:-2], forecasts[:-2]):  # Skip Naive and Drift
    res = diebold_mariano(actual, fc, fc_naive, h=1, loss="mse", one_sided=True)
    sig = "*" if res.pvalue < 0.05 else ""
    print(f"  {name:>8} vs Naive: DM={res.statistic:>7.3f}, p={res.pvalue:.4f} {sig}")

## 5. Harvey-Leybourne-Newbold Correction

The original DM statistic can be **oversized in small samples** — it rejects $H_0$ too often. Harvey, Leybourne & Newbold (1997) proposed a correction factor:

$$DM_{HLN} = DM \times \sqrt{\frac{T + 1 - 2h + h(h-1)/T}{T}}$$

This correction shrinks the statistic, making the test more conservative. The corrected statistic is compared to a $t(T-1)$ distribution instead of the standard normal.

Let's compare the original DM (without correction) to the HLN-corrected version:

In [ ]:
# Compare DM with and without HLN correction
print("DM Test: ARIMA vs ETS \u2014 HLN Correction Comparison")
print("=" * 60)

results_comparison = []
for pair_name, fc1, fc2 in [
    ("ARIMA vs ETS", fc_arima, fc_ets),
    ("ARIMA vs VAR", fc_arima, fc_var),
    ("ETS vs Naive", fc_ets, fc_naive),
]:
    res_hln = diebold_mariano(actual, fc1, fc2, h=1, loss="mse", hln_correction=True)
    res_orig = diebold_mariano(actual, fc1, fc2, h=1, loss="mse", hln_correction=False)
    results_comparison.append({
        "Pair": pair_name,
        "DM (original)": res_orig.statistic,
        "p (original)": res_orig.pvalue,
        "DM (HLN)": res_hln.statistic,
        "p (HLN)": res_hln.pvalue,
    })

comp_df = pd.DataFrame(results_comparison)
print(comp_df.to_string(index=False, float_format="%.4f"))

print("\nNote: HLN correction shrinks the statistic toward zero,")
print("making p-values larger (more conservative). The effect is")
print(f"modest here because T={len(actual)} is reasonably large.")

## 6. Multi-Horizon DM

For **multi-step-ahead forecasts** ($h > 1$), the loss differentials $d_t$ are serially correlated up to order $h-1$ (due to overlapping forecast errors). The DM test accounts for this by using HAC (Newey-West) variance with truncation lag $h-1$.

Let's see how the DM test results change across different forecast horizons:

In [ ]:
# DM test at different horizons
horizons = [1, 2, 3, 6, 12]
pairs = [
    ("ARIMA vs ETS", fc_arima, fc_ets),
    ("ARIMA vs Naive", fc_arima, fc_naive),
    ("VAR vs Drift", fc_var, fc_drift),
]

fig, axes = plt.subplots(1, len(pairs), figsize=(15, 5), sharey=True)

for ax, (pair_name, fc1, fc2) in zip(axes, pairs):
    dm_stats = []
    pvalues = []
    for h in horizons:
        res = diebold_mariano(actual, fc1, fc2, h=h, loss="mse")
        dm_stats.append(res.statistic)
        pvalues.append(res.pvalue)

    ax.bar(range(len(horizons)), pvalues, tick_label=[str(h) for h in horizons], alpha=0.7)
    ax.axhline(0.05, color="red", linestyle="--", label="alpha=0.05")
    ax.set_xlabel("Forecast Horizon (h)")
    ax.set_ylabel("p-value")
    ax.set_title(pair_name)
    ax.legend()
    ax.set_ylim(0, 1)

plt.suptitle("DM Test p-values at Different Horizons", fontsize=13)
plt.tight_layout()
plt.show()

# Detailed table for ARIMA vs ETS
print("\nDM Test: ARIMA vs ETS at Multiple Horizons")
print("-" * 55)
print(f"{'Horizon':>8} {'DM stat':>10} {'p-value':>10} {'HAC lag':>10} {'Reject?':>10}")
print("-" * 55)
for h in horizons:
    res = diebold_mariano(actual, fc_arima, fc_ets, h=h, loss="mse")
    reject = "Yes" if res.pvalue < 0.05 else "No"
    print(f"{h:>8} {res.statistic:>10.4f} {res.pvalue:>10.4f} {h - 1:>10} {reject:>10}")

## Exercise 1: Apply DM test to all M4 series

Load the `m4_sample.csv` dataset and compute pairwise DM tests for each of the 6 M4 series. Summarize how often each model "wins" across series.

In [ ]:
# Exercise 1 - DM pairwise for each M4 series
m4 = load_m4_sample()
series_ids = sorted(m4["series_id"].unique())
m4_models = ["Model1", "Model2", "Model3"]
m4_pairs = [(m4_models[i], m4_models[j]) for i in range(3) for j in range(i + 1, 3)]

print("DM Pairwise Tests for Each M4 Series")
print("=" * 70)

# Collect results in a structured table
all_results: list[dict[str, object]] = []

for sid in series_ids:
    subset = m4[m4["series_id"] == sid]
    actual_s = subset["actual"].values
    fc_dict = {
        "Model1": subset["fc_model1"].values,
        "Model2": subset["fc_model2"].values,
        "Model3": subset["fc_model3"].values,
    }

    print(f"\n--- {sid} (T={len(actual_s)}) ---")
    # MSE for reference
    for mname in m4_models:
        mse = np.mean((actual_s - fc_dict[mname]) ** 2)
        print(f"  MSE {mname}: {mse:.6f}")

    # Pairwise DM tests
    print(f"  {'Pair':<20} {'DM stat':>10} {'p-value':>10} {'Better':>10}")
    print("  " + "-" * 52)
    for m_a, m_b in m4_pairs:
        res = diebold_mariano(actual_s, fc_dict[m_a], fc_dict[m_b], h=1, loss="mse")
        better = m_a if res.mean_loss_diff < 0 else m_b
        sig = "*" if res.pvalue < 0.05 else ""
        print(f"  {m_a + ' vs ' + m_b:<20} {res.statistic:>10.4f} {res.pvalue:>10.4f} {better:>10} {sig}")
        all_results.append({
            "series": sid, "pair": f"{m_a} vs {m_b}",
            "dm_stat": res.statistic, "pvalue": res.pvalue,
            "better": better, "significant": res.pvalue < 0.05,
        })

# Summary table of p-values across series
results_df = pd.DataFrame(all_results)
print("\n\nSummary: p-value Table (series x pair)")
print("=" * 60)
pivot = results_df.pivot(index="series", columns="pair", values="pvalue")
print(pivot.round(4).to_string())

# Count wins per model
print("\n\nModel Win Count (significant DM at 5%)")
print("-" * 40)
sig_results = results_df[results_df["significant"]]
if len(sig_results) > 0:
    win_counts = sig_results["better"].value_counts()
    for model, count in win_counts.items():
        print(f"  {model}: {count} significant wins")
else:
    print("  No significant differences found at 5% level.")

print("\nDiscussion: The heterogeneity in results across M4 series illustrates")
print("that no single model dominates uniformly. The DM test may reject for some")
print("series but not others, reflecting genuine differences in forecast difficulty")
print("and model suitability across different data-generating processes.")

## Exercise 2: How does sample size affect DM test power?

Using the inflation forecasts data, run DM tests on subsamples of increasing size (e.g., T=20, 40, 60, 80, 100, 120). Plot how the p-value changes as the sample grows. At what sample size does the test first reject $H_0$?

In [ ]:
# Exercise 2 - DM test with subsamples of increasing size
sample_sizes = [30, 50, 70, 90, 110]

# Test two pairs: one with a large true difference (ARIMA vs Naive) and one
# with a small difference (ARIMA vs ETS)
test_pairs = [
    ("ARIMA vs Naive", fc_arima, fc_naive),
    ("ARIMA vs ETS", fc_arima, fc_ets),
    ("VAR vs Drift", fc_var, fc_drift),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: p-values vs sample size
ax = axes[0]
for pair_name, fc1, fc2 in test_pairs:
    pvals = []
    for T in sample_sizes:
        res = diebold_mariano(actual[:T], fc1[:T], fc2[:T], h=1, loss="mse")
        pvals.append(res.pvalue)
    ax.plot(sample_sizes, pvals, marker="o", label=pair_name)

ax.axhline(0.05, color="red", linestyle="--", alpha=0.7, label="alpha=0.05")
ax.set_xlabel("Sample Size (T)")
ax.set_ylabel("p-value")
ax.set_title("DM p-value vs Sample Size")
ax.legend()
ax.set_ylim(-0.02, 1.02)

# Right panel: effect size (mean loss diff) vs sample size
ax = axes[1]
for pair_name, fc1, fc2 in test_pairs:
    effects = []
    for T in sample_sizes:
        res = diebold_mariano(actual[:T], fc1[:T], fc2[:T], h=1, loss="mse")
        effects.append(res.mean_loss_diff)
    ax.plot(sample_sizes, effects, marker="s", label=pair_name)

ax.axhline(0, color="gray", linestyle="-", alpha=0.3)
ax.set_xlabel("Sample Size (T)")
ax.set_ylabel("Mean Loss Differential")
ax.set_title("Effect Size vs Sample Size")
ax.legend()

plt.tight_layout()
plt.show()

# Detailed table
print("\nDetailed Results: p-value and Effect Size by Sample Size")
print("=" * 75)
for pair_name, fc1, fc2 in test_pairs:
    print(f"\n{pair_name}:")
    print(f"  {'T':>5} {'DM stat':>10} {'p-value':>10} {'Mean d':>12} {'Reject 5%?':>12}")
    print("  " + "-" * 50)
    first_reject: int | None = None
    for T in sample_sizes:
        res = diebold_mariano(actual[:T], fc1[:T], fc2[:T], h=1, loss="mse")
        reject = res.pvalue < 0.05
        if reject and first_reject is None:
            first_reject = T
        print(f"  {T:>5} {res.statistic:>10.4f} {res.pvalue:>10.4f} {res.mean_loss_diff:>12.6f} {'Yes *' if reject else 'No':>12}")
    if first_reject is not None:
        print(f"  -> First rejection at T={first_reject}")
    else:
        print(f"  -> No rejection at any sample size tested")

print("\nDiscussion:")
print("- Statistical power increases with sample size: larger T -> smaller p-values")
print("  (for pairs with a true difference in predictive accuracy).")
print("- The effect size (mean loss differential) stabilizes as T grows, converging")
print("  to the population value.")
print("- For pairs with a large true difference (e.g., ARIMA vs Naive), the DM test")
print("  rejects even at small T. For pairs with a small difference (ARIMA vs ETS),")
print("  larger samples are needed to detect the difference.")
print("- This illustrates the classic power-sample size relationship: the DM test is")
print("  consistent (power -> 1 as T -> inf) but may lack power in finite samples.")